In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys

sys.path.append("../")
from tqdm import tqdm

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_only_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_only_with_layer_mppc_spacetime.npy"

BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"


sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)

X_pixel = np.concatenate([sig_pixel_spacetime, bg_pixel_spacetime], axis=0)
X_mppc = np.concatenate([sig_mppc_spacetime, bg_mppc_spacetime], axis=0)
y = np.concatenate(
    [np.ones(sig_pixel_spacetime.shape[0]), np.zeros(bg_pixel_spacetime.shape[0])],
    axis=0,
)

from sklearn.model_selection import train_test_split


X_pixel_train, X_pixel_test, X_mppc_train, X_mppc_test, y_train, y_test = (
    train_test_split(X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y)
)


del (
    sig_pixel_spacetime,
    sig_mppc_spacetime,
    bg_pixel_spacetime,
    bg_mppc_spacetime,
    X_pixel,
    X_mppc,
    y,
)

In [2]:
import src.torch.pre_processing.graph_batching as gc
from importlib import reload

reload(gc)

from torch_geometric.data import Data, Batch, Dataset
from torch_geometric.loader import DataLoader

event_processor = gc.EventProcessor(gc.HeteroGraphBuilder())

hetero_graph_train = event_processor.process_to_graphs(
    X_pixel=X_pixel_train, X_mppc=X_mppc_train, labels=y_train
)
hetero_graph_test = event_processor.process_to_graphs(
    X_pixel=X_pixel_test, X_mppc=X_mppc_test, labels=y_test
)

train_loader = DataLoader(hetero_graph_train, batch_size=512, shuffle=True)
test_loader = DataLoader(hetero_graph_test, batch_size=512, shuffle=False)

del X_pixel_train, X_pixel_test, X_mppc_train, X_mppc_test, y_train, y_test

In [ ]:
import torch
from torch_geometric.nn import HeteroConv, SAGEConv, global_mean_pool, global_max_pool, BatchNorm
import torch.nn.functional as F
from src.torch.model.components import get_mlp


class EventEdgeHeteroGNN(torch.nn.Module):
    def __init__(self, node_dims, edge_types, hidden_dim=32, num_layers=4, dropout=0.1, aggregation_scheme : list | str | None = None):
        super(EventEdgeHeteroGNN, self).__init__()
        self.node_dims = node_dims
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        if aggregation_scheme is None:
            self.aggregation_scheme = ["mean"] * num_layers
        elif isinstance(aggregation_scheme, str):
            self.aggregation_scheme = [aggregation_scheme] * num_layers
        elif isinstance(aggregation_scheme, list):
            if len(aggregation_scheme) != num_layers:
                raise ValueError("Length of aggregation_scheme list must match num_layers")
            self.aggregation_scheme = aggregation_scheme
        else:
            raise ValueError("aggregation_scheme must be None, a string, or a list of strings")

        # Initial linear transformations for each node type
        self.node_lin = torch.nn.ModuleDict({
            node_type: get_mlp(in_dim, hidden_dim, num_layers=3, dropout=dropout)
            for node_type, in_dim in node_dims.items()
        })

        # Initial edge weight learning for each edge type (if needed)
        self.edge_lin = torch.nn.ModuleDict({
            edge_type: torch.nn.Sequential(
                get_mlp(1, hidden_dim, num_layers=3, dropout=dropout),
                torch.nn.Sigmoid()
            )
            for edge_type in edge_types
        })


        # HeteroConv layers + per-node-type batchnorms
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        for conv_layer_index in range(num_layers):
            conv = HeteroConv(
                {
                    edge_type: SAGEConv((hidden_dim, hidden_dim), hidden_dim)
                    for edge_type in edge_types
                },
                aggr=aggregation_scheme[conv_layer_index]# <--- changed from "sum"
            )
            self.convs.append(conv)

            # BatchNorm for each node type in this layer
            self.bns.append(torch.nn.ModuleDict({
                node_type: BatchNorm(hidden_dim) for node_type in node_dims
            }))

        # Final linear layer for classification
        self.classifier = get_mlp(
            2 * hidden_dim * len(node_dims), 1, num_layers=3, dropout=dropout
        )

    def forward(self, input_data):
        x_dict, edge_index_dict, batch_dict = (
            input_data.x_dict,
            input_data.edge_index_dict,
            input_data.batch_dict,
        )
        if set(x_dict.keys()) != set(self.node_dims.keys()):
            print(f"Expected node types: {self.node_dims.keys()}")
            print(f"Received node types: {x_dict.keys()}")
            raise ValueError("Node types in input do not match model configuration.")

        # Initial node feature transformation
        x_dict = {
            node_type: torch.relu(self.node_lin[node_type](x))
            for node_type, x in x_dict.items()
        }

        # Apply HeteroConv layers
        for layer, conv in enumerate(self.convs):
            x_dict = conv(x_dict, edge_index_dict)

            # Apply BN + ReLU + Dropout per node type
            new_x_dict = {}
            for node_type, x in x_dict.items():
                x = self.bns[layer][node_type](x)   # BN first
                x = torch.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
                new_x_dict[node_type] = x
            x_dict = new_x_dict

        # Global pooling (mean + max for each node type)
        pooled = []
        for node_type, x in x_dict.items():
            pooled.append(global_mean_pool(x, batch_dict[node_type]))
            pooled.append(global_max_pool(x, batch_dict[node_type]))

        # Concatenate pooled features from all node types
        h = torch.cat(pooled, dim=1)

        # Classification 
        out = self.classifier(h).squeeze()
        return torch.sigmoid(out).squeeze(-1)
